In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer

import re
from bs4 import BeautifulSoup

from spacy.lang.en.stop_words import STOP_WORDS
import en_core_web_sm

from pathlib import Path
from scipy.sparse import save_npz, load_npz
import joblib


1. Importer

In [2]:
df = pd.read_csv(r'C:\Users\LN\Documents\Jedha\Projets\NORTH_FACE\Helene\data\sample_clean.csv')

In [3]:
pd.set_option('display.max_colwidth', None)
df.head(1)

,id,products,advantages,description_text,details,fabric,weight,country
0,1,Active classic boxers,"There's a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. The quick-drying, lightweight underwear takes up minimal space in a travel pack. An exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. Made of 3.7-oz 100% recycled polyester with moisture-wicking performance. Inseam (size M) is 4 1/2"". Recyclable through the Common Threads Recycling Program.","active classic boxers - there s a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. the quick-drying, lightweight underwear takes up minimal space in a travel pack. an exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. made of 3.7-oz 100% recycled polyester with moisture-wicking performance. inseam size m is 4 1/2 . recyclable through the common threads recycling program.","silky capilene 1 fabric is ultralight, breathable and quick-to-dry exposed, brushed elastic waistband for comfort 5-panel construction with traditional boxer back inseam size m is 4 1/2",3.7-oz 100% all-recycled polyester with gladiodor natural odor control for the garment. recyclable through the common threads recycling program,99 g 3.5 oz,Mexico


In [4]:
# sample = data.sample(n=5, random_state=42)
df = df.copy()

In [5]:
print(df.shape)

(500, 8)


In [6]:
cols = ['products', 'advantages']
[print(c , type(c)) for c in cols]

products <class 'str'>
advantages <class 'str'>


[None, None]

In [7]:
df['text'] = df['products'] + ' ' + df['advantages']  + ' ' + df['details'] + ' ' + df['fabric']

2. tokenizer

In [8]:
nlp = en_core_web_sm.load()

df["tokenized_text"] = df['text'].apply(
    lambda text: " ".join(
        token.text for token in nlp(text) if not token.is_stop
    ) if pd.notna(text) else text
)

In [9]:
df.head(1)

,id,products,advantages,description_text,details,fabric,weight,country,text,tokenized_text
0,1,Active classic boxers,"There's a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. The quick-drying, lightweight underwear takes up minimal space in a travel pack. An exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. Made of 3.7-oz 100% recycled polyester with moisture-wicking performance. Inseam (size M) is 4 1/2"". Recyclable through the Common Threads Recycling Program.","active classic boxers - there s a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. the quick-drying, lightweight underwear takes up minimal space in a travel pack. an exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. made of 3.7-oz 100% recycled polyester with moisture-wicking performance. inseam size m is 4 1/2 . recyclable through the common threads recycling program.","silky capilene 1 fabric is ultralight, breathable and quick-to-dry exposed, brushed elastic waistband for comfort 5-panel construction with traditional boxer back inseam size m is 4 1/2",3.7-oz 100% all-recycled polyester with gladiodor natural odor control for the garment. recyclable through the common threads recycling program,99 g 3.5 oz,Mexico,"Active classic boxers There's a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. The quick-drying, lightweight underwear takes up minimal space in a travel pack. An exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. Made of 3.7-oz 100% recycled polyester with moisture-wicking performance. Inseam (size M) is 4 1/2"". Recyclable through the Common Threads Recycling Program. silky capilene 1 fabric is ultralight, breathable and quick-to-dry exposed, brushed elastic waistband for comfort 5-panel construction with traditional boxer back inseam size m is 4 1/2 3.7-oz 100% all-recycled polyester with gladiodor natural odor control for the garment. recyclable through the common threads recycling program","Active classic boxers reason boxers cult favorite - cool , especially sticky situations . quick - drying , lightweight underwear takes minimal space travel pack . exposed , brushed waistband offers - - skin softness , - panel construction traditional boxer classic fit , functional fly . 3.7 - oz 100 % recycled polyester moisture - wicking performance . Inseam ( size M ) 4 1/2 "" . Recyclable Common Threads Recycling Program . silky capilene 1 fabric ultralight , breathable quick - - dry exposed , brushed elastic waistband comfort 5 - panel construction traditional boxer inseam size m 4 1/2 3.7 - oz 100 % - recycled polyester gladiodor natural odor control garment . recyclable common threads recycling program"


In [10]:
docs = []
for text in df["text"].dropna():
    doc = nlp(text)
    print("Tokens:", [token.text for token in doc])
    print("Lemma:", [token.lemma_ for token in doc])
    print("Indices:", [token.i for token in doc])
    print("Alpha Numeric:", [token.is_alpha for token in doc])
    print("Numbers:", [token.like_num for token in doc])
    print("Stop words restants:", [token.text for token in doc if token.is_stop])
    docs.append(doc)

Tokens: ['Active', 'classic', 'boxers', 'There', "'s", 'a', 'reason', 'why', 'our', 'boxers', 'are', 'a', 'cult', 'favorite', '-', 'they', 'keep', 'their', 'cool', ',', 'especially', 'in', 'sticky', 'situations', '.', 'The', 'quick', '-', 'drying', ',', 'lightweight', 'underwear', 'takes', 'up', 'minimal', 'space', 'in', 'a', 'travel', 'pack', '.', 'An', 'exposed', ',', 'brushed', 'waistband', 'offers', 'next', '-', 'to', '-', 'skin', 'softness', ',', 'five', '-', 'panel', 'construction', 'with', 'a', 'traditional', 'boxer', 'back', 'for', 'a', 'classic', 'fit', ',', 'and', 'a', 'functional', 'fly', '.', 'Made', 'of', '3.7', '-', 'oz', '100', '%', 'recycled', 'polyester', 'with', 'moisture', '-', 'wicking', 'performance', '.', 'Inseam', '(', 'size', 'M', ')', 'is', '4', '1/2', '"', '.', 'Recyclable', 'through', 'the', 'Common', 'Threads', 'Recycling', 'Program', '.', 'silky', 'capilene', '1', 'fabric', 'is', 'ultralight', ',', 'breathable', 'and', 'quick', '-', 'to', '-', 'dry', 'expos

In [11]:
docs

[Active classic boxers There's a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. The quick-drying, lightweight underwear takes up minimal space in a travel pack. An exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. Made of 3.7-oz 100% recycled polyester with moisture-wicking performance. Inseam (size M) is 4 1/2". Recyclable through the Common Threads Recycling Program. silky capilene 1 fabric is ultralight, breathable and quick-to-dry exposed, brushed elastic waistband for comfort 5-panel construction with traditional boxer back inseam size m is 4 1/2 3.7-oz 100% all-recycled polyester with gladiodor natural odor control for the garment. recyclable through the common threads recycling program,
 Active sport boxer briefs Skinning up Glory requires enough movement without your boxers deciding to poach their own route. The form-fitting A

In [12]:
print(STOP_WORDS)

{'wherever', 'seemed', 'throughout', 'hereupon', 'thereafter', 'whoever', 'she', 'are', 'themselves', 'besides', 'during', '‘re', '’d', 'further', 'bottom', 'ourselves', 'no', 'when', 'of', 'from', 'seeming', 'toward', 'twenty', 'next', 'everywhere', 'yourselves', 'above', 'several', 'am', 'thereupon', 'therefore', 'around', 'some', 'fifty', '’m', 'fifteen', 'someone', 'regarding', 'latterly', 'few', 'four', 'meanwhile', 'anyhow', 'former', 'much', 'both', 'front', 'than', 'onto', 'same', 'too', 'an', 'through', 'still', 'with', 'make', 'using', 'might', 'whence', 'among', '‘ll', 'everything', 'has', 'last', 'can', 'about', 'afterwards', 'namely', "'ll", 'first', 'rather', 'so', 'top', 'does', 'my', 'otherwise', 'alone', 'else', 'one', 'until', 'seems', 'show', 'if', 'which', 'nor', 'beside', 'itself', 'nine', 'it', 'was', 'often', 'becomes', 'unless', 'others', 'yourself', 'latter', '‘m', 'against', 'therein', 'on', 'had', 'yet', 'who', 'take', 'somewhere', 'her', 'due', 'where', "'d"

3. Lemmatiser

In [13]:
df["lemmatized_text"] = [
    " ".join(
        token.lemma_.lower()
        for token in doc
        if not token.is_stop and token.is_alpha
    )
    for doc in nlp.pipe(df['text'], batch_size=100)
]

In [14]:
pd.set_option('display.max_colwidth', None)
df[["text", "tokenized_text", "lemmatized_text"]].head(1)

,text,tokenized_text,lemmatized_text
0,"Active classic boxers There's a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. The quick-drying, lightweight underwear takes up minimal space in a travel pack. An exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. Made of 3.7-oz 100% recycled polyester with moisture-wicking performance. Inseam (size M) is 4 1/2"". Recyclable through the Common Threads Recycling Program. silky capilene 1 fabric is ultralight, breathable and quick-to-dry exposed, brushed elastic waistband for comfort 5-panel construction with traditional boxer back inseam size m is 4 1/2 3.7-oz 100% all-recycled polyester with gladiodor natural odor control for the garment. recyclable through the common threads recycling program","Active classic boxers reason boxers cult favorite - cool , especially sticky situations . quick - drying , lightweight underwear takes minimal space travel pack . exposed , brushed waistband offers - - skin softness , - panel construction traditional boxer classic fit , functional fly . 3.7 - oz 100 % recycled polyester moisture - wicking performance . Inseam ( size M ) 4 1/2 "" . Recyclable Common Threads Recycling Program . silky capilene 1 fabric ultralight , breathable quick - - dry exposed , brushed elastic waistband comfort 5 - panel construction traditional boxer inseam size m 4 1/2 3.7 - oz 100 % - recycled polyester gladiodor natural odor control garment . recyclable common threads recycling program",active classic boxer reason boxer cult favorite cool especially sticky situation quick dry lightweight underwear take minimal space travel pack expose brush waistband offer skin softness panel construction traditional boxer classic fit functional fly oz recycled polyester moisture wicke performance inseam size m recyclable common threads recycling program silky capilene fabric ultralight breathable quick dry expose brush elastic waistband comfort panel construction traditional boxer inseam size m oz recycle polyester gladiodor natural odor control garment recyclable common thread recycle program


4. Vectoriser + Exporter

In [15]:
configurations = {
    "unigram": {
        "ngram_range": (1, 1),
        "sublinear_tf": False
    },
    "bigram": {
        "ngram_range": (1, 2),
        "sublinear_tf": False
    },
    "sublinear": {
        "ngram_range": (1, 1),
        "sublinear_tf": True
    }
}

In [16]:
for config_name, config in configurations.items():
    print(f"Configuration : {config_name}")  
    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_df=0.95,
        min_df=2,
        **config
    )

    # Remplace les valeurs manquantes par une chaîne vide
    tfidf_matrix = vectorizer.fit_transform(df["lemmatized_text"].fillna(""))

    print("shape :", tfidf_matrix.shape)
    print("sample_words :", vectorizer.get_feature_names_out()[:20])

    processed_dir = Path(r'C:\Users\LN\Documents\Jedha\Projets\NORTH_FACE\Helene\data\processed')
    models_dir = Path("../models")

    processed_dir.mkdir(parents=True, exist_ok=True)
    models_dir.mkdir(parents=True, exist_ok=True)

    # Matrice TF-IDF
    save_npz(processed_dir / f"tfidf_{config_name}_matrix.npz", tfidf_matrix)

    # Vectoriseur déjà entraîné
    joblib.dump(vectorizer, models_dir / f"tfidf_{config_name}_vectorizer.joblib")

    # Correspondance : ligne de la matrice → produit
    df[["id"]].to_csv(
        processed_dir / f"tfidf_row_ids.csv",
        index=False,
    )

Configuration : unigram
shape : (500, 2036)
sample_words : ['ability' 'abrasion' 'absorb' 'absorption' 'abuse' 'abusive' 'access'
 'accessory' 'accommodate' 'accomplish' 'accord' 'act' 'action' 'active'
 'activity' 'adapt' 'adaptability' 'adaptable' 'add' 'addition']
Configuration : bigram
shape : (500, 8059)
sample_words : ['ability' 'abrasion' 'abrasion articulate' 'abrasion flat'
 'abrasion protect' 'abrasion resistance' 'abrasion resistant'
 'abrasion slide' 'abrasion update' 'absorb' 'absorb high' 'absorption'
 'abuse' 'abuse cargo' 'abuse new' 'abusive' 'abusive tendency' 'access'
 'access document' 'access drop']
Configuration : sublinear
shape : (500, 2036)
sample_words : ['ability' 'abrasion' 'absorb' 'absorption' 'abuse' 'abusive' 'access'
 'accessory' 'accommodate' 'accomplish' 'accord' 'act' 'action' 'active'
 'activity' 'adapt' 'adaptability' 'adaptable' 'add' 'addition']


In [17]:
X_classique = load_npz(
    processed_dir / "tfidf_unigram_matrix.npz"
)
X_sublinear = load_npz(
    processed_dir / "tfidf_sublinear_matrix.npz"
)

difference = X_classique - X_sublinear
difference.eliminate_zeros()

max_variance = (
    np.abs(difference.data).max()
    if difference.nnz else 0.0
)

print("Écart maximal entre les poids :", max_variance)
print(
    "Matrices identiques à la tolérance choisie :",
    np.allclose(difference.data, 0, atol=1e-10, rtol=0)
)

Écart maximal entre les poids : 0.2650780248044329
Matrices identiques à la tolérance choisie : False


In [21]:
import os
dossier = r"C:\Users\LN\Documents\Jedha\Projets\NORTH_FACE\Helene\data\processed"
print(os.path.exists(dossier))
print(os.listdir(dossier))

True
['sample_clean.csv', 'tfidf_bigram_matrix.npz', 'tfidf_row_ids.csv', 'tfidf_sublinear_matrix.npz', 'tfidf_unigram_matrix.npz']


In [19]:
df.head(1)

,id,products,advantages,description_text,details,fabric,weight,country,text,tokenized_text,lemmatized_text
0,1,Active classic boxers,"There's a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. The quick-drying, lightweight underwear takes up minimal space in a travel pack. An exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. Made of 3.7-oz 100% recycled polyester with moisture-wicking performance. Inseam (size M) is 4 1/2"". Recyclable through the Common Threads Recycling Program.","active classic boxers - there s a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. the quick-drying, lightweight underwear takes up minimal space in a travel pack. an exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. made of 3.7-oz 100% recycled polyester with moisture-wicking performance. inseam size m is 4 1/2 . recyclable through the common threads recycling program.","silky capilene 1 fabric is ultralight, breathable and quick-to-dry exposed, brushed elastic waistband for comfort 5-panel construction with traditional boxer back inseam size m is 4 1/2",3.7-oz 100% all-recycled polyester with gladiodor natural odor control for the garment. recyclable through the common threads recycling program,99 g 3.5 oz,Mexico,"Active classic boxers There's a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. The quick-drying, lightweight underwear takes up minimal space in a travel pack. An exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. Made of 3.7-oz 100% recycled polyester with moisture-wicking performance. Inseam (size M) is 4 1/2"". Recyclable through the Common Threads Recycling Program. silky capilene 1 fabric is ultralight, breathable and quick-to-dry exposed, brushed elastic waistband for comfort 5-panel construction with traditional boxer back inseam size m is 4 1/2 3.7-oz 100% all-recycled polyester with gladiodor natural odor control for the garment. recyclable through the common threads recycling program","Active classic boxers reason boxers cult favorite - cool , especially sticky situations . quick - drying , lightweight underwear takes minimal space travel pack . exposed , brushed waistband offers - - skin softness , - panel construction traditional boxer classic fit , functional fly . 3.7 - oz 100 % recycled polyester moisture - wicking performance . Inseam ( size M ) 4 1/2 "" . Recyclable Common Threads Recycling Program . silky capilene 1 fabric ultralight , breathable quick - - dry exposed , brushed elastic waistband comfort 5 - panel construction traditional boxer inseam size m 4 1/2 3.7 - oz 100 % - recycled polyester gladiodor natural odor control garment . recyclable common threads recycling program",active classic boxer reason boxer cult favorite cool especially sticky situation quick dry lightweight underwear take minimal space travel pack expose brush waistband offer skin softness panel construction traditional boxer classic fit functional fly oz recycled polyester moisture wicke performance inseam size m recyclable common threads recycling program silky capilene fabric ultralight breathable quick dry expose brush elastic waistband comfort panel construction traditional boxer inseam size m oz recycle polyester gladiodor natural odor control garment recyclable common thread recycle program


In [20]:
df.to_csv(r'C:\Users\LN\Documents\Jedha\Projets\NORTH_FACE\Helene\data\processed\sample_clean.csv', index=False)